### Project: A/B Testing Analysis

Problem: We introduce a new feature to increase conversion rate and want to assess if the feature is effective.

I designed and analyzed an A/B test to evaluate the impact of a new feature on user conversation rate.
I estimated treatment effects using statistical inference and confidence intervals.
I interpreted results under uncertainty and recommended rollout decision based on practical significance.

I designed the dataset that contains features:
user_id
group (control / treatment)
outcome (converted or not)

The Primary metric I selected:
conversion rate = % of users who convert

In [15]:
# Dataset Generation

# control conversion rate = 10%
# treatment = 12%

import numpy as np
import pandas as pd

np.random.seed(42)

# Parameters
n_users = 10000
control_rate = 0.10
treatment_rate = 0.12

# Assign users randomly
user_ids = np.arange(n_users)
groups = np.random.choice(['control', 'treatment'], size=n_users)


# Generate conversions
conversions = []

for g in groups:
    if g == 'control':
        conversions.append(np.random.binomial(1, control_rate))
    else:
        conversions.append(np.random.binomial(1, treatment_rate))


# Create dataset

df = pd.DataFrame({
    'user_id': user_ids,
    'group': groups,
    'converted': conversions
})

# Preview
print(df.head())


   user_id      group  converted
0        0    control          0
1        1  treatment          0
2        2    control          0
3        3    control          0
4        4    control          0


In [26]:
# Print how many users in control and treatment groups

print(df.groupby('group')['user_id'].nunique())

group
control      5013
treatment    4987
Name: user_id, dtype: int64


In [27]:
# To test if there a Sample-Ratio Mismatch, I applied chi-square goodness of fit test.

from scipy.stats import chisquare

chi2_stat, p_val = chisquare(f_obs=[5013, 4987], f_exp=[5000, 5000])

print(f"Chi-square Stat: {chi2_stat}, P-value: {p_val}")



Chi-square Stat: 0.0676, P-value: 0.794863773596479


Since the p-value is much larger than 0.05, not statistically significant, we conclude that there is no Sample-Ratio Mismatch and the difference between the number of users in treatment and control groups can be explained by random variations. 

In [28]:
# Print the converted rates for control and treatment groups

print(df.groupby('group')['converted'].mean())

group
control      0.100339
treatment    0.118709
Name: converted, dtype: float64


In [29]:
# Compute the converted rates for control and treatment groups and lift

converted_control = df[df['group'] == 'control']['converted']
converted_treatment = df[df['group'] == 'treatment']['converted']

control_rate = converted_control.mean()
treatment_rate = converted_treatment.mean()
lift = treatment_rate - control_rate

print("control rate: ",control_rate)
print("treatment rate", treatment_rate)
print("lift: ", lift)

control rate:  0.10033911829243966
treatment rate 0.1187086424704231
lift:  0.018369524177983446


In [30]:
# Statistical test: a two-proportion z-test is used to compute p-value

from statsmodels.stats.proportion import proportions_ztest

count = [treatment.sum(), control.sum()]
nobs = [len(treatment), len(control)]

stat, pval = proportions_ztest(count, nobs)

print("stat:", stat)
print("p-value:", pval)

stat: 2.9413209883130333
p-value: 0.003268156792524678


In [31]:
# Confidence interval

p_t = count[0] / nobs[0]
p_c = count[1] / nobs[1]

diff = p_t - p_c

se = np.sqrt(
    p_t * (1 - p_t) / nobs[0] +
    p_c * (1 - p_c) / nobs[1]
)

z = 1.96  # for 95% CI

ci_low = diff - z * se
ci_high = diff + z * se

print(ci_low, ci_high)

0.00613162894429194 0.03060741941167495


### INTERPRETATION
The treatment group shows a higher conversion rate (12%) compared to control (10%), corresponding to a lift of 2 percentage points.
The difference is statistically significant (p < 0.05), suggesting the observed effect is unlikely due to random variation. We also compute  the 95% confidence interval to assess the effect size.


### DECISION

Given the statistically significant improvement and meaningful lift, 
I would recommend rolling out the feature, while continuing to monitor performance for potential long-term effects.